# Front RF-DETR ready4: ONNX → TensorRT → COCO 평가

이 노트북은 `notebook-front-ready4.tar.gz` 하나를 업로드하여 다음 작업을 순서대로 수행합니다.

1. 아카이브와 내부 파일의 SHA-256 검증 및 안전한 압축 해제
2. NVIDIA GPU, CUDA, PyTorch, TensorRT 버전 점검
3. B01 FP32 → B02 FP16 → B03 INT8 PTQ → R01 FP16 엔진 생성
4. 각 엔진을 분리된 test 437장으로 bbox AP, mask AP, semantic mIoU 평가
5. 엔진·메타데이터·평가 결과를 ZIP으로 다운로드

Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 먼저 선택하세요. 셀은 위에서 아래로 한 번씩 실행합니다. 중단 후 같은 런타임에서 다시 실행하면 검증된 기존 엔진과 평가 결과를 재사용합니다. B03 보정에는 번들에 포함된 train 128장만 사용하며 test는 최종 평가에만 사용합니다.


## 1. 아카이브 업로드

아래 셀은 `/content/notebook-front-ready4.tar.gz`가 이미 있고 SHA가 맞으면 다시 업로드하지 않습니다. 그렇지 않으면 파일 선택 창에서 전달받은 아카이브를 선택합니다.


In [ ]:
from pathlib import Path
import hashlib

EXPECTED_ARCHIVE_SHA256 = "9b4a9ed3bb1e097351e143f6e6d03031ef15f802e055cd17d2637ffdc827a722"
ARCHIVE = Path("/content/notebook-front-ready4.tar.gz")
ROOT = Path("/content/notebook-front-ready4")

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if ARCHIVE.is_file() and file_sha256(ARCHIVE) == EXPECTED_ARCHIVE_SHA256:
    print(f"기존 아카이브 재사용: {ARCHIVE}")
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("notebook-front-ready4.tar.gz 파일 하나만 업로드하세요.")
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    uploaded_path = Path("/content") / Path(uploaded_name).name
    if uploaded_path != ARCHIVE:
        uploaded_path.replace(ARCHIVE)
    del uploaded, uploaded_bytes

actual_archive_sha256 = file_sha256(ARCHIVE)
if actual_archive_sha256 != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError(
        "아카이브 SHA-256 불일치. 잘못되었거나 손상된 파일입니다.\n"
        f"expected={EXPECTED_ARCHIVE_SHA256}\nactual={actual_archive_sha256}"
    )
print(f"아카이브 SHA-256 확인 완료: {actual_archive_sha256}")


## 2. 안전한 압축 해제와 전체 payload 검증

경로 이탈, 심볼릭 링크, 장치 파일을 허용하지 않고 일반 파일과 디렉터리만 `/content` 아래에 풉니다. 이어서 ONNX, 568개 데이터 payload, COCO 주석과 split 역할을 검증합니다.


In [ ]:
import json
import os
import shutil
import tarfile
from concurrent.futures import ThreadPoolExecutor

def safe_extract_regular_files(archive_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with tarfile.open(archive_path, "r:gz") as archive:
        members = archive.getmembers()
        if not members:
            raise RuntimeError("빈 아카이브입니다.")
        for member in members:
            relative = Path(member.name)
            if relative.is_absolute() or ".." in relative.parts:
                raise RuntimeError(f"안전하지 않은 아카이브 경로: {member.name}")
            if not relative.parts or relative.parts[0] != ROOT.name:
                raise RuntimeError(f"예상하지 않은 최상위 경로: {member.name}")
            target = (destination / relative).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"압축 해제 경로 이탈: {member.name}")
            if member.issym() or member.islnk() or member.isdev() or member.isfifo():
                raise RuntimeError(f"허용되지 않는 아카이브 항목: {member.name}")
            if not (member.isdir() or member.isfile()):
                raise RuntimeError(f"지원하지 않는 아카이브 항목: {member.name}")
        for member in members:
            target = destination / member.name
            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            source = archive.extractfile(member)
            if source is None:
                raise RuntimeError(f"아카이브 파일을 읽을 수 없음: {member.name}")
            temporary = target.with_name(target.name + ".extracting")
            with source, temporary.open("wb") as output:
                shutil.copyfileobj(source, output, length=4 * 1024 * 1024)
            os.replace(temporary, target)

safe_extract_regular_files(ARCHIVE, Path("/content"))
print(f"안전한 압축 해제 완료: {ROOT}")

MANIFEST_PATH = ROOT / "manifest.json"
DATA_MANIFEST_PATH = ROOT / "data-manifest.json"
CHECKSUMS_PATH = ROOT / "data-checksums.sha256"
for required in (MANIFEST_PATH, DATA_MANIFEST_PATH, CHECKSUMS_PATH):
    if not required.is_file():
        raise FileNotFoundError(required)

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
data_manifest = json.loads(DATA_MANIFEST_PATH.read_text(encoding="utf-8"))
expected_order = ["B01", "B02", "B03", "R01"]
actual_order = [item["experiment_id"] for item in sorted(manifest["engine_plan"], key=lambda item: item["order"])]
if manifest.get("suite") != "ready4" or actual_order != expected_order:
    raise RuntimeError(f"ready4 엔진 순서 불일치: {actual_order}")
if manifest.get("calibration_dir") != "data/training/front_session_split_v1/train":
    raise RuntimeError("B03 calibration 경로는 train split이어야 합니다.")

def inside_root(relative_text: str) -> Path:
    relative = Path(relative_text)
    if relative.is_absolute() or ".." in relative.parts:
        raise RuntimeError(f"안전하지 않은 manifest 경로: {relative_text}")
    resolved = (ROOT / relative).resolve()
    if ROOT.resolve() not in resolved.parents:
        raise RuntimeError(f"manifest 경로 이탈: {relative_text}")
    return resolved

for source in manifest["onnx_sources"]:
    path = inside_root(source["path"])
    if not path.is_file() or path.stat().st_size != int(source["size_bytes"]):
        raise RuntimeError(f"ONNX 크기 불일치: {path}")
    if file_sha256(path) != source["sha256"]:
        raise RuntimeError(f"ONNX SHA-256 불일치: {path}")

checksum_lines = [line for line in CHECKSUMS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
checksum_jobs = []
for line in checksum_lines:
    expected_hash, relative_text = line.split("  ", 1)
    checksum_jobs.append((expected_hash, inside_root(relative_text)))

def verify_one(job):
    expected_hash, path = job
    if not path.is_file():
        return f"누락: {path}"
    actual_hash = file_sha256(path)
    return None if actual_hash == expected_hash else f"SHA 불일치: {path}"

with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 1)) as executor:
    checksum_errors = [error for error in executor.map(verify_one, checksum_jobs) if error]
if checksum_errors:
    raise RuntimeError("payload 검증 실패:\n" + "\n".join(checksum_errors[:20]))

IMAGE_SUFFIXES = {".bmp", ".jpeg", ".jpg", ".png", ".webp"}
SPLIT_ROOT = ROOT / "data/training/front_session_split_v1"
TRAIN_DIR = SPLIT_ROOT / "train"
TEST_DIR = SPLIT_ROOT / "test"
train_images = sorted(path for path in TRAIN_DIR.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)
test_images = sorted(path for path in TEST_DIR.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)
selected_names = [line.strip() for line in (ROOT / "calibration-selection.txt").read_text(encoding="utf-8").splitlines() if line.strip()]
if len(train_images) != 128 or [path.name for path in train_images] != sorted(selected_names):
    raise RuntimeError("INT8 보정 데이터가 고정된 train 128장과 일치하지 않습니다.")
test_coco = json.loads((TEST_DIR / "_annotations.coco.json").read_text(encoding="utf-8"))
if len(test_images) != 437 or len(test_coco.get("images", [])) != 437 or len(test_coco.get("annotations", [])) != 593:
    raise RuntimeError("test split의 437장/593개 주석 조건이 맞지 않습니다.")
if set(path.name for path in train_images) & set(path.name for path in test_images):
    raise RuntimeError("train calibration과 test 사이 파일명 중복이 있습니다.")
if not data_manifest.get("leakage_verification", {}).get("passed"):
    raise RuntimeError("데이터 누수 검증을 통과하지 않은 번들입니다.")
print(f"payload SHA-256 {len(checksum_jobs)}개 확인 완료")
print(f"INT8 calibration: train {len(train_images)}장 전용")
print(f"최종 평가: test {len(test_images)}장, annotation {len(test_coco['annotations'])}개 전용")


## 3. 실행 계획 dry-run

실제 GPU 작업 전에 manifest의 입력 ONNX, 출력 engine, precision, 입력 크기와 보정 경로를 출력합니다. 이 셀은 엔진을 만들지 않습니다.


In [ ]:
SOURCE_BY_ID = {item["experiment_id"]: inside_root(item["path"]) for item in manifest["onnx_sources"]}
PLAN_BY_ID = {item["experiment_id"]: item for item in manifest["engine_plan"]}
for experiment_id in expected_order:
    item = PLAN_BY_ID[experiment_id]
    source = SOURCE_BY_ID[item["source_experiment_id"]]
    target = ROOT / "artifacts/experiments" / experiment_id / "front/model.engine"
    calibration = TRAIN_DIR if item["precision"] == "int8" else None
    print({
        "order": item["order"],
        "experiment": experiment_id,
        "precision": item["precision"],
        "input_shape": item["input_shape"],
        "source_onnx": str(source),
        "target_engine": str(target),
        "calibration": str(calibration) if calibration else None,
    })


## 4. 고정 버전 설치

프로젝트 기준 버전을 설치합니다. 이미 같은 버전이면 pip가 재사용합니다. 이 셀 실행 중 CUDA/PyTorch 관련 패키지 다운로드로 시간이 걸릴 수 있습니다. 설치가 끝난 뒤 다음 셀에서 실제 로드된 버전을 다시 검사합니다.


In [ ]:
import subprocess
import sys

PINNED_PACKAGES = [
    "torch==2.10.0",
    "torchvision==0.25.0",
    "rfdetr[onnx]==1.8.1",
    "onnx==1.22.0",
    "pycocotools==2.0.11",
    "numpy==2.2.6",
    "Pillow==12.3.0",
    "PyYAML==6.0.2",
    "tensorrt-cu13==10.16.1.11",
]
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", *PINNED_PACKAGES], check=True)
print("고정 패키지 설치 완료")


## 5. GPU·CUDA·TensorRT·ONNX 사전 점검

GPU 런타임이 아니거나 실제 import 버전이 프로젝트 기준과 다르면 엔진 생성 전에 명확한 오류로 중단합니다. Colab이 이전 모듈을 메모리에 유지해 버전이 다르게 보인다면 **런타임 → 세션 다시 시작** 후 이 노트북을 위에서부터 다시 실행하세요.


In [ ]:
import importlib.metadata as metadata
import platform
from datetime import datetime, timezone

import onnx
import tensorrt as trt
import torch
import torchvision

expected_versions = {
    "torch": "2.10.0",
    "torchvision": "0.25.0",
    "rfdetr": "1.8.1",
    "onnx": "1.22.0",
    "pycocotools": "2.0.11",
    "numpy": "2.2.6",
    "Pillow": "12.3.0",
    "tensorrt-cu13": "10.16.1.11",
}
version_errors = []
for package_name, expected_version in expected_versions.items():
    try:
        actual_version = metadata.version(package_name).split("+", 1)[0]
    except metadata.PackageNotFoundError:
        version_errors.append(f"{package_name}: 설치되지 않음 (expected {expected_version})")
        continue
    if actual_version != expected_version:
        version_errors.append(f"{package_name}: {actual_version} (expected {expected_version})")
if trt.__version__ != "10.16.1.11":
    version_errors.append(f"TensorRT module: {trt.__version__} (expected 10.16.1.11)")
if version_errors:
    raise RuntimeError(
        "프로젝트 고정 버전과 현재 런타임이 다릅니다. 런타임을 다시 시작하고 재실행하세요.\n- "
        + "\n- ".join(version_errors)
    )
if platform.system() != "Linux" or platform.machine() not in {"x86_64", "AMD64"}:
    raise RuntimeError(f"TensorRT wheel 대상이 아닌 플랫폼입니다: {platform.platform()}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다. Colab 런타임 유형을 GPU로 변경하세요.")
if not torch.version.cuda:
    raise RuntimeError("CUDA 지원 PyTorch가 로드되지 않았습니다.")
logger = trt.Logger(trt.Logger.ERROR)
if trt.Builder(logger) is None:
    raise RuntimeError("TensorRT Builder 초기화 실패: CUDA/driver/TensorRT 조합을 확인하세요.")

for source in SOURCE_BY_ID.values():
    model = onnx.load(str(source), load_external_data=True)
    onnx.checker.check_model(model)

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout.strip()
environment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "platform": platform.platform(),
    "python": platform.python_version(),
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": list(torch.cuda.get_device_capability(0)),
    "nvidia_smi": smi,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "torchvision": torchvision.__version__,
    "tensorrt": trt.__version__,
    "onnx": onnx.__version__,
}
ENVIRONMENT_PATH = ROOT / "results/environment.json"
ENVIRONMENT_PATH.parent.mkdir(parents=True, exist_ok=True)
ENVIRONMENT_PATH.write_text(json.dumps(environment, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(environment, indent=2, ensure_ascii=False))
print("ONNX checker 통과")


## 6. TensorRT 빌드 함수 준비

manifest를 유일한 엔진 계획으로 사용합니다. 생성 완료 후 source SHA, GPU, compute capability와 TensorRT 버전이 모두 같은 엔진은 재사용합니다. B03은 검증된 train 128장만 읽습니다.


In [ ]:
import tempfile
import time

import torchvision.transforms.functional as functional
from PIL import Image

WORKSPACE_MIB = 4096
FORCE_REBUILD = False
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def engine_paths(experiment_id: str):
    directory = ROOT / "artifacts/experiments" / experiment_id / "front"
    return directory / "model.engine", directory / "engine-build.json", directory / "calibration.cache"

def reusable_engine(experiment_id: str, source_sha: str, precision: str) -> bool:
    engine_path, metadata_path, _ = engine_paths(experiment_id)
    if FORCE_REBUILD or not engine_path.is_file() or not metadata_path.is_file():
        return False
    try:
        record = json.loads(metadata_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    expected = {
        "source_onnx_sha256": source_sha,
        "precision": precision,
        "gpu": environment["gpu"],
        "compute_capability": environment["compute_capability"],
        "tensorrt": environment["tensorrt"],
    }
    return all(record.get(key) == value for key, value in expected.items()) and record.get("engine_sha256") == file_sha256(engine_path)

def parse_network(builder, logger, onnx_path: Path, expected_shape):
    flags = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    network = builder.create_network(flags)
    parser = trt.OnnxParser(network, logger)
    if not parser.parse(onnx_path.read_bytes()):
        errors = "\n".join(str(parser.get_error(index)) for index in range(parser.num_errors))
        raise RuntimeError(f"TensorRT ONNX parse 실패 ({onnx_path}):\n{errors}")
    if network.num_inputs != 1:
        raise RuntimeError(f"ONNX 입력은 하나여야 합니다: {network.num_inputs}")
    actual_shape = [int(value) for value in network.get_input(0).shape]
    if actual_shape != list(expected_shape):
        raise RuntimeError(f"입력 크기 불일치: actual={actual_shape}, expected={expected_shape}")
    return network

def make_calibrator_class():
    class TrainOnlyImageCalibrator(trt.IInt8EntropyCalibrator2):
        def __init__(self, images, input_shape, cache_path):
            super().__init__()
            if len(images) != 128 or any(TEST_DIR in path.parents for path in images):
                raise RuntimeError("INT8 calibration은 train 128장만 허용합니다.")
            self.images = images
            self.input_shape = tuple(input_shape)
            self.cache_path = cache_path
            self.index = 0
            self.device_input = torch.empty(self.input_shape, dtype=torch.float32, device="cuda")

        def get_batch_size(self):
            return self.input_shape[0]

        def get_batch(self, names):
            del names
            if self.index >= len(self.images):
                return None
            batch_size, _, height, width = self.input_shape
            batch_paths = self.images[self.index:self.index + batch_size]
            if len(batch_paths) != batch_size:
                return None
            tensors = []
            for path in batch_paths:
                with Image.open(path) as image:
                    tensor = functional.to_tensor(image.convert("RGB"))
                tensor = functional.resize(tensor, [height, width], antialias=True)
                tensors.append(functional.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD))
            self.device_input.copy_(torch.stack(tensors))
            self.index += batch_size
            print(f"\rcalibration: {self.index}/{len(self.images)}", end="", flush=True)
            return [int(self.device_input.data_ptr())]

        def read_calibration_cache(self):
            return self.cache_path.read_bytes() if self.cache_path.is_file() else None

        def write_calibration_cache(self, cache):
            self.cache_path.parent.mkdir(parents=True, exist_ok=True)
            self.cache_path.write_bytes(cache)
    return TrainOnlyImageCalibrator

def build_engine(experiment_id: str) -> Path:
    item = PLAN_BY_ID[experiment_id]
    precision = item["precision"]
    source = SOURCE_BY_ID[item["source_experiment_id"]]
    source_sha = file_sha256(source)
    engine_path, metadata_path, cache_path = engine_paths(experiment_id)
    engine_path.parent.mkdir(parents=True, exist_ok=True)
    if reusable_engine(experiment_id, source_sha, precision):
        print(f"{experiment_id}: 검증된 기존 engine 재사용 ({engine_path})")
        return engine_path

    logger = trt.Logger(trt.Logger.INFO)
    builder = trt.Builder(logger)
    network = parse_network(builder, logger, source, item["input_shape"])
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, WORKSPACE_MIB * 1024 * 1024)
    calibrator = None
    if precision == "fp16":
        config.set_flag(trt.BuilderFlag.FP16)
    elif precision == "int8":
        config.set_flag(trt.BuilderFlag.INT8)
        config.set_flag(trt.BuilderFlag.FP16)
        Calibrator = make_calibrator_class()
        calibrator = Calibrator(train_images, item["input_shape"], cache_path)
        config.int8_calibrator = calibrator
    elif precision != "fp32":
        raise RuntimeError(f"지원하지 않는 precision: {precision}")

    started = time.perf_counter()
    serialized = builder.build_serialized_network(network, config)
    del calibrator
    if serialized is None:
        raise RuntimeError(f"{experiment_id} TensorRT engine 생성 실패")
    temporary = engine_path.with_suffix(".engine.building")
    temporary.write_bytes(serialized)
    os.replace(temporary, engine_path)
    elapsed = time.perf_counter() - started
    record = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_id": experiment_id,
        "source_experiment_id": item["source_experiment_id"],
        "source_onnx": str(source.relative_to(ROOT)),
        "source_onnx_sha256": source_sha,
        "precision": precision,
        "input_shape": item["input_shape"],
        "workspace_mib": WORKSPACE_MIB,
        "calibration_split": "train" if precision == "int8" else None,
        "calibration_image_count": 128 if precision == "int8" else 0,
        "engine": str(engine_path.relative_to(ROOT)),
        "engine_size_bytes": engine_path.stat().st_size,
        "engine_sha256": file_sha256(engine_path),
        "build_seconds": elapsed,
        "gpu": environment["gpu"],
        "compute_capability": environment["compute_capability"],
        "torch": environment["torch"],
        "torch_cuda": environment["torch_cuda"],
        "tensorrt": environment["tensorrt"],
    }
    metadata_path.write_text(json.dumps(record, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"{experiment_id}: {precision} engine 생성 완료 ({elapsed:.1f}s, {engine_path.stat().st_size / 2**20:.1f} MiB)")
    return engine_path


## 7. 엔진 순차 생성

GPU 메모리 충돌을 피하고 비교 조건을 고정하기 위해 아래 네 셀을 순서대로 실행합니다. 이미 성공한 셀은 재실행해도 같은 런타임에서 해당 엔진을 재사용합니다.


In [ ]:
B01_ENGINE = build_engine("B01")  # 1/4: baseline FP32


In [ ]:
B02_ENGINE = build_engine("B02")  # 2/4: baseline FP16


In [ ]:
B03_ENGINE = build_engine("B03")  # 3/4: train 128장 INT8 PTQ


In [ ]:
R01_ENGINE = build_engine("R01")  # 4/4: 432x432 FP16


## 8. TensorRT 추론·COCO 평가 함수 준비

저장소의 평가 방식과 동일하게 confidence `0.001`로 bbox/mask AP를 계산하고, confidence `0.25` 이상 mask의 클래스별 semantic union으로 dataset mIoU를 계산합니다.


In [ ]:
from collections import defaultdict

import numpy as np
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from rfdetr.models.postprocess import PostProcess
from supervision import Detections

class TensorRTRunner:
    def __init__(self, engine_path: Path):
        self.trt = trt
        self.engine_path = engine_path
        self.logger = trt.Logger(trt.Logger.ERROR)
        self.runtime = trt.Runtime(self.logger)
        self.engine = self.runtime.deserialize_cuda_engine(engine_path.read_bytes())
        if self.engine is None:
            raise RuntimeError(f"TensorRT engine 역직렬화 실패: {engine_path}")
        self.context = self.engine.create_execution_context()
        if self.context is None:
            raise RuntimeError(f"TensorRT execution context 생성 실패: {engine_path}")
        self.input_name = ""
        self.outputs = {}
        for index in range(self.engine.num_io_tensors):
            name = self.engine.get_tensor_name(index)
            shape = tuple(self.engine.get_tensor_shape(name))
            if any(dimension < 0 for dimension in shape):
                raise RuntimeError(f"동적 tensor shape은 지원하지 않습니다: {name}={shape}")
            numpy_dtype = np.dtype(trt.nptype(self.engine.get_tensor_dtype(name)))
            torch_dtype = torch.from_numpy(np.empty((), dtype=numpy_dtype)).dtype
            tensor = torch.empty(shape, dtype=torch_dtype, device="cuda")
            self.context.set_tensor_address(name, tensor.data_ptr())
            if self.engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT:
                if self.input_name:
                    raise RuntimeError("TensorRT 입력은 하나여야 합니다.")
                self.input_name = name
                self.input = tensor
            else:
                self.outputs[name] = tensor
        if not self.input_name or not {"dets", "labels", "masks"}.issubset(self.outputs):
            raise RuntimeError("TensorRT 출력 dets/labels/masks를 찾을 수 없습니다.")
        if self.input.ndim != 4 or self.input.shape[0] != 1:
            raise RuntimeError(f"고정 NCHW batch-1 입력이 필요합니다: {tuple(self.input.shape)}")
        output_shapes = {name: tuple(tensor.shape) for name, tensor in self.outputs.items()}
        if any(len(output_shapes[name]) < 2 for name in ("dets", "labels", "masks")):
            raise RuntimeError(f"RF-DETR 출력 rank가 잘못되었습니다: {output_shapes}")
        query_counts = {name: output_shapes[name][1] for name in ("dets", "labels", "masks")}
        if len(set(query_counts.values())) != 1 or next(iter(query_counts.values())) < 1:
            raise RuntimeError(f"RF-DETR 출력 query 수가 일치하지 않습니다: {query_counts}")
        self.query_count = next(iter(query_counts.values()))
        self.postprocess_num_select = self.query_count
        self.postprocess = PostProcess(num_select=self.postprocess_num_select)

    def predict(self, image: Image.Image, threshold: float) -> Detections:
        source_image = np.array(image)
        original_width, original_height = image.size
        _, _, input_height, input_width = self.input.shape
        tensor = functional.to_tensor(image).to(device="cuda", dtype=self.input.dtype)
        tensor = functional.resize(tensor, [input_height, input_width], antialias=True)
        tensor = functional.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)
        self.input.copy_(tensor.unsqueeze(0))
        stream = torch.cuda.current_stream()
        if not self.context.execute_async_v3(stream.cuda_stream):
            raise RuntimeError("TensorRT execute_async_v3 실패")
        results = self.postprocess(
            {"pred_boxes": self.outputs["dets"], "pred_logits": self.outputs["labels"], "pred_masks": self.outputs["masks"]},
            target_sizes=torch.tensor([[original_height, original_width]], device="cuda"),
        )
        result = results[0]
        keep = result["scores"] > threshold
        detections = Detections(
            xyxy=result["boxes"][keep].float().cpu().numpy(),
            confidence=result["scores"][keep].float().cpu().numpy(),
            class_id=result["labels"][keep].cpu().numpy(),
            mask=result["masks"][keep].squeeze(1).cpu().numpy(),
        )
        detections.metadata["source_image"] = source_image
        detections.data["source_shape"] = np.tile(np.array([original_height, original_width], dtype=np.int64), (len(detections), 1))
        return detections

def annotation_mask(annotation, height, width):
    segmentation = annotation.get("segmentation")
    if not segmentation:
        return np.zeros((height, width), dtype=bool)
    if isinstance(segmentation, dict):
        rle = dict(segmentation)
        if isinstance(rle.get("counts"), list):
            rle = mask_utils.frPyObjects(rle, height, width)
        elif isinstance(rle.get("counts"), str):
            rle["counts"] = rle["counts"].encode("ascii")
    else:
        rle = mask_utils.frPyObjects(segmentation, height, width)
        if isinstance(rle, list):
            rle = mask_utils.merge(rle)
    return mask_utils.decode(rle).astype(bool)

def summarize_coco(evaluation):
    names = ("ap", "ap50", "ap75", "ap_small", "ap_medium", "ap_large", "ar1", "ar10", "ar100", "ar_small", "ar_medium", "ar_large")
    return {name: float(value) for name, value in zip(names, evaluation.stats, strict=True)}

def result_is_reusable(experiment_id, engine_path, output_path):
    if not output_path.is_file():
        return False
    try:
        result = json.loads(output_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return (
        result.get("experiment_id") == experiment_id
        and result.get("engine_sha256") == file_sha256(engine_path)
        and result.get("annotation_sha256") == data_manifest["test"]["annotation_sha256"]
        and result.get("image_count") == 437
        and result.get("threshold") == 0.001
        and result.get("miou_threshold") == 0.25
    )

def evaluate_engine(experiment_id: str):
    engine_path, _, _ = engine_paths(experiment_id)
    output_dir = ROOT / "results/coco-evaluation"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{experiment_id}-front.json"
    if result_is_reusable(experiment_id, engine_path, output_path):
        print(f"{experiment_id}: 검증된 437장 평가 결과 재사용")
        return json.loads(output_path.read_text(encoding="utf-8"))

    coco = COCO(str(TEST_DIR / "_annotations.coco.json"))
    image_ids = sorted(coco.getImgIds())
    category_ids = sorted(category_id for category_id in coco.getCatIds() if coco.getAnnIds(catIds=[category_id]))
    runner = TensorRTRunner(engine_path)
    bbox_predictions, segmentation_predictions = [], []
    intersections, unions = defaultdict(int), defaultdict(int)
    started = time.perf_counter()
    for index, image_id in enumerate(image_ids, start=1):
        info = coco.loadImgs([image_id])[0]
        image_path = TEST_DIR / info["file_name"]
        if not image_path.is_file():
            image_path = TEST_DIR / "images" / info["file_name"]
        with Image.open(image_path) as opened:
            image = opened.convert("RGB")
        detections = runner.predict(image, threshold=0.001)
        masks = detections.mask
        predicted_semantic = {category_id: np.zeros((info["height"], info["width"]), dtype=bool) for category_id in category_ids}
        for detection_index in range(len(detections)):
            category_id = int(detections.class_id[detection_index])
            if category_id not in category_ids:
                continue
            common = {"image_id": image_id, "category_id": category_id, "score": float(detections.confidence[detection_index])}
            x1, y1, x2, y2 = (float(value) for value in detections.xyxy[detection_index])
            bbox_predictions.append({**common, "bbox": [x1, y1, x2 - x1, y2 - y1]})
            if masks is not None:
                mask = masks[detection_index].astype(bool)
                encoded = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
                encoded["counts"] = encoded["counts"].decode("ascii")
                segmentation_predictions.append({**common, "segmentation": encoded})
                if common["score"] >= 0.25:
                    predicted_semantic[category_id] |= mask
        ground_truth_semantic = {category_id: np.zeros((info["height"], info["width"]), dtype=bool) for category_id in category_ids}
        for annotation in coco.loadAnns(coco.getAnnIds(imgIds=[image_id])):
            category_id = int(annotation["category_id"])
            if category_id in ground_truth_semantic:
                ground_truth_semantic[category_id] |= annotation_mask(annotation, info["height"], info["width"])
        for category_id in category_ids:
            prediction, target = predicted_semantic[category_id], ground_truth_semantic[category_id]
            intersections[category_id] += int(np.count_nonzero(prediction & target))
            unions[category_id] += int(np.count_nonzero(prediction | target))
        print(f"\r{experiment_id} test: {index}/{len(image_ids)}", end="", flush=True)
    print()
    if not bbox_predictions or not segmentation_predictions:
        raise RuntimeError(f"{experiment_id}: bbox 또는 mask 예측이 없습니다.")

    def run_coco(predictions, iou_type):
        detected = coco.loadRes(predictions)
        evaluation = COCOeval(coco, detected, iou_type)
        evaluation.params.imgIds = image_ids
        evaluation.params.catIds = category_ids
        evaluation.evaluate(); evaluation.accumulate(); evaluation.summarize()
        return summarize_coco(evaluation)

    bbox_metrics = run_coco(bbox_predictions, "bbox")
    segmentation_metrics = run_coco(segmentation_predictions, "segm")
    category_ious = {str(key): intersections[key] / unions[key] for key in category_ids if unions[key]}
    semantic_miou = float(np.mean(list(category_ious.values())))
    result = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_id": experiment_id,
        "camera": "front",
        "engine": str(engine_path.relative_to(ROOT)),
        "engine_sha256": file_sha256(engine_path),
        "engine_size_bytes": engine_path.stat().st_size,
        "dataset": str(TEST_DIR.relative_to(ROOT)),
        "annotation_sha256": data_manifest["test"]["annotation_sha256"],
        "image_count": len(image_ids),
        "category_ids": category_ids,
        "threshold": 0.001,
        "miou_threshold": 0.25,
        "evaluation_seconds": time.perf_counter() - started,
        "hardware": environment,
        "prediction_count": len(bbox_predictions),
        "metrics": {"bbox": bbox_metrics, "segm": segmentation_metrics, "semantic_miou": semantic_miou, "semantic_iou_by_category": category_ious},
    }
    output_path.write_text(json.dumps(result, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"{experiment_id}: bbox AP={bbox_metrics['ap']:.6f}, mask AP={segmentation_metrics['ap']:.6f}, mIoU={semantic_miou:.6f}")
    return result


## 9. 네 엔진 smoke test

437장 전체 평가 전에 동일한 실제 test 이미지 한 장으로 네 엔진의 역직렬화와 추론을 확인합니다.


In [ ]:
smoke_image_path = test_images[0]
with Image.open(smoke_image_path) as opened:
    smoke_image = opened.convert("RGB")
for experiment_id in expected_order:
    engine_path, _, _ = engine_paths(experiment_id)
    runner = TensorRTRunner(engine_path)
    detections = runner.predict(smoke_image, threshold=0.5)
    torch.cuda.synchronize()
    if runner.postprocess_num_select != runner.query_count:
        raise RuntimeError(f"{experiment_id}: PostProcess num_select와 engine Q 불일치")
    print(f"{experiment_id}: input={tuple(runner.input.shape)}, Q={runner.query_count}, num_select={runner.postprocess_num_select}, detections={len(detections)}")
    del runner
    torch.cuda.empty_cache()


## 10. test 437장 최종 평가

정확한 비교를 위해 네 엔진을 같은 test split과 같은 threshold로 순차 평가합니다. test 이미지는 어느 calibration 단계에서도 사용되지 않았습니다.


In [ ]:
evaluation_results = {}
for experiment_id in expected_order:
    evaluation_results[experiment_id] = evaluate_engine(experiment_id)
    torch.cuda.empty_cache()


## 11. 결과 요약


In [ ]:
summary_rows = []
for experiment_id in expected_order:
    result = evaluation_results[experiment_id]
    summary_rows.append({
        "experiment": experiment_id,
        "precision": PLAN_BY_ID[experiment_id]["precision"],
        "input_shape": PLAN_BY_ID[experiment_id]["input_shape"],
        "engine_mib": round(result["engine_size_bytes"] / 2**20, 2),
        "bbox_ap": result["metrics"]["bbox"]["ap"],
        "mask_ap": result["metrics"]["segm"]["ap"],
        "semantic_miou": result["metrics"]["semantic_miou"],
        "evaluation_seconds": result["evaluation_seconds"],
    })
SUMMARY_PATH = ROOT / "results/ready4-summary.json"
SUMMARY_PATH.write_text(json.dumps(summary_rows, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
try:
    import pandas as pd
    display(pd.DataFrame(summary_rows))
except ImportError:
    print(json.dumps(summary_rows, indent=2, ensure_ascii=False))


## 12. 엔진과 결과 압축·다운로드

engine, engine build metadata, B03 calibration cache, 평가 JSON, 실행 환경과 원본 manifest만 ZIP에 담습니다. ONNX와 이미지 데이터는 중복 포함하지 않습니다.


In [ ]:
import zipfile

OUTPUT_ZIP = Path("/content/front-ready4-tensorrt-results.zip")
temporary_zip = OUTPUT_ZIP.with_suffix(".zip.building")
export_files = [MANIFEST_PATH, DATA_MANIFEST_PATH, ENVIRONMENT_PATH, SUMMARY_PATH]
for experiment_id in expected_order:
    engine_path, metadata_path, cache_path = engine_paths(experiment_id)
    result_path = ROOT / "results/coco-evaluation" / f"{experiment_id}-front.json"
    export_files.extend([engine_path, metadata_path, result_path])
    if cache_path.is_file():
        export_files.append(cache_path)
for path in export_files:
    if not path.is_file():
        raise FileNotFoundError(path)
with zipfile.ZipFile(temporary_zip, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as bundle:
    for path in export_files:
        bundle.write(path, path.relative_to(ROOT).as_posix())
os.replace(temporary_zip, OUTPUT_ZIP)
print(f"결과 ZIP: {OUTPUT_ZIP}")
print(f"크기: {OUTPUT_ZIP.stat().st_size / 2**20:.1f} MiB")
print(f"SHA-256: {file_sha256(OUTPUT_ZIP)}")
from google.colab import files
files.download(str(OUTPUT_ZIP))
